# Compare peri-trigger firing histogram max times

This notebook peri-trigger firing histograms between spike-types, within groups.

First, for each cell, collects the specified triggers and calculates the firing histogram around the given time window around each trigger for each spike-type separately. After smoothing and averaging the histograms, the time of the maximal firing frequency is detected for each spike type.

Finally, a paired hypothesis test is performed within a group of cells to test the difference between spike types.

## Initialization - don't change

In [ ]:
import sys
sys.path.append('..')

from workflows.run_util.io import collect_paths_from_table
from param_classes.workflows.peritrigger_analysis import PeriTriggerAnalysisWorkflowParams

from workflows.peritrigger_analysis import run_analysis, run_aggregation, run_testing

params = PeriTriggerAnalysisWorkflowParams()

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [ ]:
params.palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'RA-AP': '#00ff00'
    }
}

## Data settings

Set raw input dir. You need to give the base path where the group directories can be found.

In [ ]:
params.data.raw_data.path='E:\\Data\\BogiSejtek\\20251007\\data\\invivo'
params.data.raw_data.extension='mat'

Set the base path of the directory where csv files containing the spike features and categories can be found.
> It needs to be the same structure as the `raw_data` directory.

In [ ]:
params.data.spike_features.path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\SpikeFeatures'
params.data.spike_features.extension = 'csv'

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [ ]:
params.input = {
    'mode': 'table',
    'table_path': 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
params.output.path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\PeriTriggerAnalysis'

## Analysis parameters

First, you can set the peritrigger analysis parameters.
 - `trigger`: marker name that you want to use for triggering
 - `spike_types`: name of the spike types that you want to compare
 - `smooth_size`: the `sigma` parameter of the gaussian kernel for smoothing

In [ ]:
params.analysis.trigger = 'delta_end'
params.analysis.spike_types = ['AIS-AP', 'RA-AP']
params.analysis.smooth_size = 5

Here you can set the histogram parameters.
 - `data_unit`: time unit that you want to use. You have to give the following parameters in this unit too. Possible values: `'s'` or `'ms'`
 - `value_unit`: unit of the histogram, possible values: `'count'`, `'freq'`, `'prob'`
 - `bin_range`: range around the trigger to analyze
 - `bin_size`: size of the bin

In [ ]:
params.analysis.histogram.data_unit = 'ms'
params.analysis.histogram.value_unit = 'freq'
params.analysis.histogram.bin_range = (-400, 400)
params.analysis.histogram.bin_size = 4
params.analysis.ylim = (0, 20)

Here you can set the parameters that determine, how to detect maximal peak of the firing rate histogram around the trigger.

 - `trigger_condition`: if you leave empty (`[]`), all triggers will be used. If you put values in the list, only those triggers will be used for the analysis where that spike type was present
 - `detection_start`: where to start the search for the maximal value, relative to the trigger. Set to 0 if you want to start from the trigger.

In [ ]:
params.grouping.trigger_condition = ['RA-AP']
params.grouping.detection_start = -100
params.grouping.boxplot_ylim = (0, 400)

## Run analysis

Collect input files according to the table.

In [ ]:
input_files = collect_paths_from_table(params.data, params.input)
print(f"{len(input_files)} files selected for analysis.")

Perform individual analysis and save the results.

In [ ]:
trigger_results = run_analysis(input_files, params)

Sort cells into groups by the `group` column of the Excel table.

In [ ]:
group_names = list(set([file.info.cell_type for file in input_files]))
groups = []
for group_name in group_names:
    group_files = [
        file for file in trigger_results
        if file.info.cell_type == group_name
    ]
    groups.append((group_name, group_files))
print(f"{len(groups)} groups found for aggregation:")
for group in groups:
    print(f"\t- {group[0]}")

Aggregate each cells within groups. It will
 - average triggered histograms that satisfies the condition
 - detect maximum time and value
 - plot and save the aggregation result

In [ ]:
group_aggregates = []
for group_name, group_files in groups:
    group_aggregates.append(run_aggregation(group_files, group_name, params))

Perform spike-type comparison within each group. It saves the resulting statistics and a boxplot for each group.

In [ ]:
run_testing(group_aggregates, params)